## `summarization.ipynb: Step 4 of the pipeline`
**Role**: Generates a text summary for each product by feeding its most helpful reviews to BART.  
**Input**: `reviews_insights.pkl`, `products_insights.pkl`.                     
**Output**: `products_summarized.pkl`.

`Be aware to change the "DIRECTORY" variable (cell 3) with the right path of the folder on Google Drive where you uploaded the dataset!`

The `.pkl` format is sensitive to the Pandas version: the version used to create the file must match the one used to read it. Since Google Colab may ship a different Pandas version than the one on your local machine, run the cell below to install the exact version used during preprocessing.

In [ ]:
!pip install pandas==3.0.1

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Set DIRECTORY to the Google Drive folder where you uploaded `reviews_insights.pkl` and `products_insights.pkl`.                  
*Examples*:
```python
# Dataset is directly in MyDrive root:
DIRECTORY = HERE / "drive" / "MyDrive"

# Dataset is inside a subfolder called "Data Science in Action":
DIRECTORY = HERE / "drive" / "MyDrive" / "Data Science in Action"
```

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)
import pandas as pd
from pathlib import Path
from transformers import pipeline
from transformers import BartForConditionalGeneration, BartTokenizerFast
import torch

# --------------------------------------------------
# Paths
# --------------------------------------------------
HERE = Path.cwd()
DIRECTORY = HERE / "drive" / "MyDrive" / "Data Science in Action"  

REVIEWS_INPUT_PATH = DIRECTORY / "reviews_insights.pkl"
PRODUCTS_INPUT_PATH = DIRECTORY / "products_insights.pkl"

CHECKPOINT_PATH = DIRECTORY / "summarization_checkpoint.pkl"
OUTPUT_PATH = DIRECTORY / "products_summarized.pkl"

`CHECKPOINT_PATH` is a temporary file saved periodically so you can resume after a Colab disconnect.

### Config. variables

In [ ]:
MODEL_NAME       = "facebook/bart-large-cnn"
MAX_INPUT_CHARS  = 2000   # how much review text to feed the model per product
CHECKPOINT_EVERY = 500    # save progress every N products
BATCH_SIZE = 8 

`MAX_INPUT_CHARS` is how many characters of reviews we feed to BART. BART has a 1024-token limit, so ~2000 chars is a safe cap before we hit it.

### Loading datasets

In [ ]:
def load_data(reviews_path: Path, products_path: Path):
    print("Loading reviews...")
    try:
        reviews_df = pd.read_pickle(reviews_path)
        print(f"  {len(reviews_df):,} reviews loaded")
    except FileNotFoundError as e:
        raise FileNotFoundError(f"Failed to load reviews. Check if the path {reviews_path} is correct.") from e
    except (ModuleNotFoundError, ImportError) as e:
            raise ImportError(f"Version mismatch loading reviews.\n"
                "Did you run the '!pip install pandas==3.0.1' cell?") from e
    
    print("Loading products...")
    try:
        products_df = pd.read_pickle(products_path)
        print(f"  {len(products_df):,} products loaded")
    except FileNotFoundError as e:
        raise FileNotFoundError(f"Failed to load products. Check if the path {products_path} is correct.") from e
    except (ModuleNotFoundError, ImportError) as e:
            raise ImportError(f"Version mismatch loading products.\n"
                "Did you run the '!pip install pandas==3.0.1' cell?") from e

    return reviews_df, products_df

### Getting the text from the product's reviews

In [ ]:
def get_product_reviews_text(asin: str, reviews_df: pd.DataFrame, max_chars: int) -> str:
    """
    For a given product:
    1. Get all its reviews
    2. Sort by helpful_vote descending (most useful reviews first)
    3. Concatenate text until max_chars is reached (the review that crosses
       the limit is truncated, so a single long review is never discarded)
    """
    product_reviews = reviews_df[reviews_df["parent_asin"] == asin]
    product_reviews = product_reviews.sort_values("helpful_vote", ascending=False)

    combined = ""
    for text in product_reviews["text"].fillna(""):
        remaining = max_chars - len(combined)
        if remaining <= 0:
            break
        combined += text[:remaining] + " "

    return combined.strip()

The most community-validated reviews (highest `helpful_vote`) go in first. If text exceeds `MAX_INPUT_CHARS`, the last review is truncated and no more are added.

### Main pipeline

In [ ]:
def run_summarization(
    reviews_input_path: Path = REVIEWS_INPUT_PATH,
    products_input_path: Path = PRODUCTS_INPUT_PATH,
    checkpoint_path: Path = CHECKPOINT_PATH,
    output_path: Path = OUTPUT_PATH,
    model_name: str = MODEL_NAME,
    max_input_chars: int = MAX_INPUT_CHARS,
    checkpoint_every: int = CHECKPOINT_EVERY
):
    reviews_df, products_df = load_data(reviews_input_path, products_input_path)
    
    if checkpoint_path.exists():
        print("Checkpoint found - resuming...")
        products_df = pd.read_pickle(checkpoint_path)
    else:
        products_df["summary"] = None

    # Load model on GPU
    print(f"Loading model: {model_name}")
    tokenizer = BartTokenizerFast.from_pretrained(model_name)
    bart_model = BartForConditionalGeneration.from_pretrained(model_name, torch_dtype=torch.float16).to("cuda")

    # Only process products that don't have a summary yet
    to_process = products_df[products_df["summary"].isna()]
    batch_texts, batch_idx, batch_flags = [], [], []
    last_checkpoint = 0

    for i, (idx, row) in enumerate(to_process.iterrows()):
        asin  = row["parent_asin"]
        text  = get_product_reviews_text(asin, reviews_df, max_input_chars)

        is_last = (i == len(to_process) - 1)

        # No review text available — skip BART and set summary to empty string directly.
        # (The pending batch is still flushed below if this is the last product.)
        if not text:
            products_df.at[idx, "summary"] = ""
            if not (is_last and batch_texts):
                continue
        else:
            batch_texts.append(text)
            batch_idx.append(idx)
            batch_flags.append(len(text) >= 100)

        if len(batch_texts) == BATCH_SIZE or (is_last and batch_texts):
            inputs = tokenizer(
                batch_texts,
                return_tensors="pt",
                max_length=1024,
                truncation=True,
                padding=True                 
            ).to("cuda")

            summary_ids = bart_model.generate(
                inputs["input_ids"],
                attention_mask=inputs["attention_mask"],   
                max_length=150,
                min_length=40,
                do_sample=False
            )

            for j, (idx_, has_text) in enumerate(zip(batch_idx, batch_flags)):
                if has_text:
                    products_df.at[idx_, "summary"] = tokenizer.decode(
                        summary_ids[j], skip_special_tokens=True
                    )
                else:
                    products_df.at[idx_, "summary"] = batch_texts[j].strip()

            batch_texts, batch_idx, batch_flags = [], [], []

            # Checkpoint (at least every `checkpoint_every` products)
            if (i + 1) - last_checkpoint >= checkpoint_every:
                products_df.to_pickle(checkpoint_path)
                last_checkpoint = i + 1
                print(f"  Checkpoint saved at {i + 1:,} products")

    # Final save and cleanup
    products_df.to_pickle(output_path)
    print(f"Done. Saved to: {output_path}")

    if checkpoint_path.exists():
        checkpoint_path.unlink()

    return products_df

In [ ]:
run_summarization()